# Extended Thinking with Tool Use

## Table of contents
- [Setup](#setup)
- [Single tool call with thinking](#single-tool-call-with-thinking)
- [Multiple tool calls with thinking](#multiple-tool-calls-with-thinking)
- [Preserving thinking blocks](#preserving-thinking-blocks)

When using extended thinking with tools, Claude shows its reasoning **before** making tool requests but does **not** output another thinking block after receiving tool results — it waits until the next non-`tool_result` user turn.

## Setup

Ensure `@anthropic-ai/sdk` is installed and `ANTHROPIC_API_KEY` is in your environment.

In [16]:
// npm install @anthropic-ai/sdk


Import the SDK and define shared constants.

In [17]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const MODEL_NAME = 'claude-sonnet-4-6';
const ANTHROPIC_API_KEY = Deno.env.get('ANTHROPIC_API_KEY') ?? '';
const MAX_TOKENS = 4000;
const BUDGET_TOKENS = 2000;

const client = new Anthropic({ apiKey: ANTHROPIC_API_KEY });
console.log('Client ready. Model:', MODEL_NAME);


Client ready. Model: claude-sonnet-4-6


Shared helper that prints thinking blocks (truncated) and the final text answer.

In [18]:
function printThinkingResponse(response: Anthropic.Message): void {
  console.log('\n==== FULL RESPONSE ====');
  for (const block of response.content) {
    if (block.type === 'thinking') {
      const t = (block as any).thinking as string;
      console.log('\nTHINKING BLOCK:');
      console.log(t.length > 500 ? t.slice(0, 500) + '...' : t);
    } else if (block.type === 'text') {
      console.log('\nFINAL ANSWER:');
      console.log(block.text);
    }
  }
  console.log('\n==== END RESPONSE ====');
}


## Single tool call with thinking

Define the weather tool schema. The `input_schema` tells Claude what parameters to provide when calling the tool.

In [19]:
const weatherTool: Anthropic.Tool[] = [{
  name: 'weather',
  description: 'Get current weather information for a location.',
  input_schema: {
    type: 'object',
    properties: { location: { type: 'string', description: 'The location to get weather for.' } },
    required: ['location']
  }
}];

console.log('Tool defined:', weatherTool[0].name);


Tool defined: weather


Mock weather database and lookup function.

In [20]:
const weatherData: Record<string, { temperature: number; condition: string }> = {
  'New York': { temperature: 72, condition: 'Sunny' },
  'London':   { temperature: 62, condition: 'Cloudy' },
  'Tokyo':    { temperature: 80, condition: 'Partly cloudy' },
  'Paris':    { temperature: 65, condition: 'Rainy' },
  'Sydney':   { temperature: 85, condition: 'Clear' },
  'Berlin':   { temperature: 60, condition: 'Foggy' },
};

function getWeather(location: string) {
  return weatherData[location] ?? { error: 'No weather data for ' + location };
}


First API call — Claude sees the tool definition and should decide to call `weather`.

In [21]:
const userQuery = "What's the weather like in Paris today?";

let singleResponse = await client.messages.create({
  model: MODEL_NAME,
  max_tokens: MAX_TOKENS,
  thinking: { type: 'enabled', budget_tokens: BUDGET_TOKENS },
  tools: weatherTool,
  messages: [{ role: 'user', content: userQuery }]
});

console.log('stop_reason:', singleResponse.stop_reason);
console.log('Thinking block present:', singleResponse.content.some(b => b.type === 'thinking'));


stop_reason: tool_use
Thinking block present: true


Build the assistant turn, preserving **all** `thinking` and `tool_use` blocks, then execute the tool.

> Omitting thinking blocks here causes an API error in the next call.

In [22]:
const singleConv: Anthropic.MessageParam[] = [{ role: 'user', content: userQuery }];

const assistantBlocks = singleResponse.content.filter(
  b => b.type === 'thinking' || b.type === 'redacted_thinking' || b.type === 'tool_use'
);
singleConv.push({ role: 'assistant', content: assistantBlocks as any });

const toolUseBlock = singleResponse.content.find(b => b.type === 'tool_use') as Anthropic.ToolUseBlock;
const location = (toolUseBlock.input as any).location as string;
const toolResult = getWeather(location);

console.log(`Tool called: weather("${location}") =>`, toolResult);


Tool called: weather("Paris") => { temperature: 65, condition: "Rainy" }


Push the tool result as a `user` turn, then send back to Claude for the final answer.

In [23]:
singleConv.push({
  role: 'user',
  content: [{ type: 'tool_result', tool_use_id: toolUseBlock.id, content: JSON.stringify(toolResult) }]
});

const finalSingleResponse = await client.messages.create({
  model: MODEL_NAME,
  max_tokens: MAX_TOKENS,
  thinking: { type: 'enabled', budget_tokens: BUDGET_TOKENS },
  tools: weatherTool,
  messages: singleConv
});

printThinkingResponse(finalSingleResponse);



==== FULL RESPONSE ====

FINAL ANSWER:
Based on the latest weather data, here's what it's like in **Paris** today:

- 🌡️ **Temperature:** 65°F (around 18°C)
- 🌧️ **Condition:** Rainy

It's a cool and rainy day in the City of Light! You might want to grab an umbrella if you're heading out. 🌂

==== END RESPONSE ====


## Multiple tool calls with thinking

Define two tools — weather and news. Claude may call them in any order or call both.

In [24]:
const multiTools: Anthropic.Tool[] = [
  {
    name: 'weather',
    description: 'Get current weather information for a location.',
    input_schema: { type: 'object', properties: { location: { type: 'string' } }, required: ['location'] }
  },
  {
    name: 'news',
    description: 'Get latest news headlines for a topic.',
    input_schema: { type: 'object', properties: { topic: { type: 'string' } }, required: ['topic'] }
  }
];

console.log('Tools defined:', multiTools.map(t => t.name).join(', '));


Tools defined: weather, news


Mock news function. The weather function is reused from the single-tool section above.

In [25]:
function getNews(topic: string): { headlines: string[] } {
  const newsData: Record<string, string[]> = {
    technology: ['New AI breakthrough announced', 'Tech company releases latest smartphone'],
    sports:     ['Local team wins championship'],
    weather:    ['Storm system developing in the Atlantic']
  };
  return { headlines: newsData[topic.toLowerCase()] ?? ['No news available for this topic'] };
}


Start the conversation with the first API call.

In [32]:
const multiQuery = "What's the weather in London, and tell me the latest technology news?";
const multiConv: Anthropic.MessageParam[] = [{ role: 'user', content: multiQuery }];

let multiResponse = await client.messages.create({
  model: MODEL_NAME,
  max_tokens: MAX_TOKENS,
  thinking: { type: 'enabled', budget_tokens: BUDGET_TOKENS },
  tools: multiTools,
  messages: multiConv
});

console.log('Initial stop_reason:', multiResponse.stop_reason);
console.log(JSON.stringify(multiResponse.content, null, 2));


Initial stop_reason: tool_use
[
  {
    "type": "thinking",
    "thinking": "The user wants two things: weather in London and latest technology news. These are independent, so I can call both simultaneously.",
    "signature": "EsACClsIDRgCKkAGYMtaeqr0jLsvknDr8CFC4UgXkqw/NE0IUErd8dwELxys0QRLo3kncnqG6y4/zvAM+hZ5ntW7NdfvnYD4PHOkMhFjbGF1ZGUtc29ubmV0LTQtNjgAEgwK9mElQeBRc+tlGboaDE+0pSgiKKuN7cJ6eyIwRhLzcxc1jxoaEDtpaGsoNNWMxL+oRzF8zbs4V9IyGJfXxLHpgip/chbNUSoiHew9KpIB3788CcJ3nkgZ1I/vUUJyJuc6wY+4NpoXzV5G9DeXhbD+h5sH5YghEnqpEF5qlih6QuFi/dKREf9F9hYM0qMOrThBClgWOWWkiNcOiWGLydmzXH0XaXAO1NlTv1r2TFUQvsH82X7sf7SS0jieYFI99/LKQBkrq1JS1sdcWsM/oErEcb95xF0Jgg8ldCWSrH7YTfkYAQ=="
  },
  {
    "type": "text",
    "text": "Sure! Let me fetch both at the same time for you!"
  },
  {
    "type": "tool_use",
    "id": "toolu_018JR5sjE2LARkwR52bkFFcp",
    "name": "weather",
    "input": {
      "location": "London"
    },
    "caller": {
      "type": "direct"
    }
  },
  {
    "type": "tool_use",
    "id": "too

Loop until `stop_reason` is no longer `'tool_use'`. Each iteration executes the requested tool and sends the result back.

In [27]:
while (multiResponse.stop_reason === 'tool_use') {
  // Preserve all thinking + tool_use blocks in assistant turn
  const blocks = multiResponse.content.filter(
    b => b.type === 'thinking' || b.type === 'redacted_thinking' || b.type === 'tool_use'
  );
  multiConv.push({ role: 'assistant', content: blocks as any });

  // Execute ALL tool calls Claude requested (it may request multiple at once)
  const toolUseBlocks = multiResponse.content.filter(b => b.type === 'tool_use') as Anthropic.ToolUseBlock[];
  const toolResults: Anthropic.ToolResultBlockParam[] = toolUseBlocks.map(tb => {
    let result: unknown;
    if (tb.name === 'weather') result = getWeather((tb.input as any).location);
    else if (tb.name === 'news') result = getNews((tb.input as any).topic);
    else result = { error: 'Unknown tool' };
    console.log(`Tool: ${tb.name}(${JSON.stringify(tb.input)}) =>`, result);
    return { type: 'tool_result' as const, tool_use_id: tb.id, content: JSON.stringify(result) };
  });

  // All tool_results MUST be in the same user message
  multiConv.push({ role: 'user', content: toolResults });

  multiResponse = await client.messages.create({
    model: MODEL_NAME, max_tokens: MAX_TOKENS,
    thinking: { type: 'enabled', budget_tokens: BUDGET_TOKENS },
    tools: multiTools, messages: multiConv
  });
}

printThinkingResponse(multiResponse);

Tool: weather({"location":"London"}) => { temperature: 62, condition: "Cloudy" }
Tool: news({"topic":"technology"}) => {
  headlines: [
    "New AI breakthrough announced",
    "Tech company releases latest smartphone"
  ]
}

==== FULL RESPONSE ====

FINAL ANSWER:
Here's the information you requested:

🌤️ **Weather in London:**
- **Temperature:** 62°F
- **Condition:** Cloudy

---

📰 **Latest Technology News:**
1. **New AI breakthrough announced** – The tech world is buzzing with the latest advancements in artificial intelligence.
2. **Tech company releases latest smartphone** – A major tech company has just unveiled its newest smartphone model.

---

Let me know if you'd like more details on either topic! 😊

==== END RESPONSE ====


## Preserving thinking blocks

Three rules when combining extended thinking with tool use:
1. **Preserve signatures** — each `thinking` block has a cryptographic signature that validates context
2. **Don't modify prior content** — the API rejects requests where previous blocks are altered
3. **Include redacted blocks too** — both `thinking` and `redacted_thinking` must stay in history

Make an initial call and capture the thinking and tool_use blocks separately.

In [28]:
const preserveQuery = "What's the weather like in Berlin right now?";

const preserveResponse = await client.messages.create({
  model: MODEL_NAME,
  max_tokens: MAX_TOKENS,
  thinking: { type: 'enabled', budget_tokens: BUDGET_TOKENS },
  tools: weatherTool,
  messages: [{ role: 'user', content: preserveQuery }]
});

const thinkingBlocks = preserveResponse.content.filter(b => b.type === 'thinking');
const toolUseBlocks  = preserveResponse.content.filter(b => b.type === 'tool_use') as Anthropic.ToolUseBlock[];

console.log('Thinking blocks:', thinkingBlocks.length, '| Tool use blocks:', toolUseBlocks.length);


Thinking blocks: 1 | Tool use blocks: 1


Intentionally omit thinking blocks when building the assistant turn — this shows the expected API error.

In [29]:
const toolBlock = toolUseBlocks[0];
const berlinWeather = getWeather((toolBlock.input as any).location);

try {
  await client.messages.create({
    model: MODEL_NAME, max_tokens: MAX_TOKENS,
    thinking: { type: 'enabled', budget_tokens: BUDGET_TOKENS },
    tools: weatherTool,
    messages: [
      { role: 'user',      content: preserveQuery },
      { role: 'assistant', content: toolUseBlocks as any },  // ← thinking blocks missing!
      { role: 'user',      content: [{ type: 'tool_result', tool_use_id: toolBlock.id, content: JSON.stringify(berlinWeather) }] }
    ]
  });
} catch (e) {
  console.log('Expected error (thinking blocks omitted):', String(e).slice(0, 200));
}


{
  model: "claude-sonnet-4-6",
  id: "msg_01YTBK5VeXTyToGCu6LgDPx3",
  type: "message",
  role: "assistant",
  content: [
    {
      type: "text",
      text: "Right now in **Berlin**, the weather is **foggy** with a temperature of **60°F (approximately 15.5°C)**. It might be a good idea to carry a jacket if you're heading out, and be cautious if you're driving due to the reduced visibility from the fog!"
    }
  ],
  stop_reason: "end_turn",
  stop_sequence: null,
  stop_details: null,
  usage: {
    input_tokens: 653,
    cache_creation_input_tokens: 0,
    cache_read_input_tokens: 0,
    cache_creation: { ephemeral_5m_input_tokens: 0, ephemeral_1h_input_tokens: 0 },
    output_tokens: 67,
    service_tier: "standard",
    inference_geo: "global"
  }
}

Correct approach: include **both** `thinking` and `tool_use` blocks in the assistant turn.

In [30]:
const completeBlocks = [...thinkingBlocks, ...toolUseBlocks];

const correctResponse = await client.messages.create({
  model: MODEL_NAME, max_tokens: MAX_TOKENS,
  thinking: { type: 'enabled', budget_tokens: BUDGET_TOKENS },
  tools: weatherTool,
  messages: [
    { role: 'user',      content: preserveQuery },
    { role: 'assistant', content: completeBlocks as any },  // ← thinking preserved ✓
    { role: 'user',      content: [{ type: 'tool_result', tool_use_id: toolBlock.id, content: JSON.stringify(berlinWeather) }] }
  ]
});

printThinkingResponse(correctResponse);



==== FULL RESPONSE ====

FINAL ANSWER:
Right now in **Berlin**, the weather is **foggy** with a temperature of **60°F (around 15°C)**. It might be a good idea to bring a jacket if you're heading out, and visibility could be reduced due to the fog. 🌫️

==== END RESPONSE ====
